# An End-to-End AI Harvest Planner for Low-Cost Fruit-Picking Robots Built on a Physics-Consistent World Model

**`02_integrated_model.ipynb`**

Four learned components and the chain that joins them. They are built here in the order the
problem builds up, which is not the order a reader would guess.

**Underneath everything, the outcome model** (§1). `P(result | observation, aperture)` over six
classes. It is not one of the three layers so much as the surface they all stand on: the station
planner ranks positions by the expected utility it implies, the pick policy takes its
probabilities as input, and the cluster search enumerates orders against it. Change this and
every number above it moves.

**Layer 1 — the settled-pose dynamics** (§2). What a pick does to the fruit it leaves behind.
Before anything can be planned, the state has to be predictable. The physics settles most of the
question on its own — a fruit with nothing holding it hangs plumb — and what remains is a twelve
per cent minority worth a classifier.

**Layer 2 — ordering inside a cluster** (§3). Exhaustive search over the orders within a cluster,
kept as a benchmark rather than shipped. Greedy matches the optimum on three quarters of clusters
at a mean gap of 0.064, and that measurement is what says ordering is not where the value is.

**Layer 3 — planning a tree** (§4). A station planner choosing where the base stops and a pick
policy choosing what to take from each stop. These two are the deliverable: detections in, a
harvest plan out.

Sections 5 to 7 join them, take one component out at a time, and record what each cost to train.

Every training step is timed and every fitted model is compared against the one shipped in
`models/`, so this notebook answers two questions at once — can it be reproduced, and how long
does it take. Training is behind flags; the default verifies rather than refits.


In [1]:
import os
import sys
import time
import json
import pickle
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Default to the folder this notebook sits in, so a clone runs without setup. An absolute
# default only ever pointed at one machine, and an environment variable set in a shell does
# not reach a kernel that was already running.
ROOT = Path(os.environ.get("AIPICK_ROOT") or Path.cwd())
os.environ["AIPICK_ROOT"] = str(ROOT)

SRC, DATA, MODELS = ROOT/"src", ROOT/"data", ROOT/"models"
TX, RUNS = DATA/"transitions", ROOT/"runs"
RUNS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC))

TREES_USED = "trees_measured_pose.csv"
TREES_TRAIN, TREES_EVAL = range(0, 20), range(20, 50)

# The arm, the stop count and the selection threshold are defined once, in src/planner.py, and
# read from there. Restating them in each notebook is how the three drifted apart: this one
# still said three stops and a 0.9 threshold long after the figures had moved on. The import
# costs nothing here -- the weights are loaded later, in section 4.

RUN_TRAIN_OUTCOME  = False    # ~1 min    LightGBM on the pick dataset
RUN_TRAIN_DYNAMICS = False    # ~10 s     CatBoost on the transitions
RUN_TRAIN_PLANNER  = False    # ~11 min   archive/training/station_planner2.ipynb
RUN_TRAIN_POLICY   = False    # ~2 h      archive/training/pick_policy5.ipynb

TIMING = {}


def timed(name):
    '''Context manager that records wall time into TIMING.'''
    class _T:
        def __enter__(self):
            self.t0 = time.time(); return self

        def __exit__(self, *a):
            TIMING[name] = time.time() - self.t0
            print(f"  {name}: {TIMING[name]:.1f} s")
    return _T()


import environment as E
E.load(ROOT, trees=TREES_USED, dynamics=True)

import planner as PL
HALF_X, LIFT = PL.HALF_X, PL.LIFT
THRESHOLD, K_STATIONS = PL.THRESHOLD, PL.K_STATIONS
print(f"environment loaded: {len(E.T):,} fruit, {len(E.FEATS)} features, "
      f"{len(E.CLASSES)} classes")
print(f"  dynamics {'on' if E.DYN else 'OFF'}"
      + (f", AUC {E.DYN['cv_auc']:.4f}" if E.DYN else ""))
print(f"  utility {dict(zip(E.CLASSES, np.round(E.UTIL_VEC, 2)))}")






environment loaded: 6,000 fruit, 15 features, 6 classes
  dynamics on, AUC 0.9299
  utility {'APPROACH_BLOCKED': np.float64(0.0), 'DEFECT': np.float64(-0.3), 'GRASP_FAILED': np.float64(0.0), 'NEIGHBOR_KNOCKED': np.float64(-0.5), 'NO_DETACH': np.float64(0.0), 'SUCCESS': np.float64(1.0)}


## 1. Layer 1 — the outcome model

`P(result | observation, aperture)` over six classes, fitted on the pick dataset. Everything
above it consumes this: the planner ranks positions by the expected utility it implies, the pick
policy takes its probabilities as input, and the cluster search enumerates orders against it.

Three of the six raw physics codes are folded into `DEFECT` at fitting time, because the utility
table gives all three the same −0.3 — the fruit is off the tree and no longer premium, and which
tissue tore is not a decision the planner makes.

Refitting is compared against the shipped model rather than trusted: same rows, same features,
and the two should agree on nearly every row. A disagreement means the recipe recorded here has
drifted from the one that produced `models/outcome.pkl`.


In [2]:
DS = pd.read_csv(DATA/"picks.csv")
DAMAGE = ("STEM_PULL", "STALK_SNAP", "SPUR_BREAK")
DS["y"] = DS.code.map(lambda c: "DEFECT" if c in DAMAGE else c)

need = [c for c in E.FEATS if c not in DS.columns]
for c, base in (("obs_nearest_diam_c", "obs_nearest_diam"),
                ("obs_nb_along_c", "obs_nb_along"),
                ("obs_nb_radial_c", "obs_nb_radial")):
    if c in need and base in DS.columns:
        v = DS[base].to_numpy(float)
        DS[c] = np.where(v == E.SENTINEL, E.FILLS[base], v)
if "obs_has_neighbour" in need and "obs_n_neighbours" in DS.columns:
    DS["obs_has_neighbour"] = (DS.obs_n_neighbours.to_numpy(float) > 0).astype(float)
still = [c for c in E.FEATS if c not in DS.columns]
assert not still, f"cannot rebuild {still}"

X, y = DS[E.FEATS].to_numpy(float), DS.y.to_numpy()
print(f"{len(DS):,} rows, {DS.apple_id.nunique():,} fruit" if "apple_id" in DS else f"{len(DS):,} rows")

if RUN_TRAIN_OUTCOME:
    import lightgbm as lgb
    tr = DS.split != "test" if "split" in DS else np.ones(len(DS), bool)
    with timed("outcome model"):
        m = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
                               min_child_samples=30, subsample=0.9, subsample_freq=1,
                               colsample_bytree=0.9, verbose=-1, random_state=0)
        m.fit(X[tr.to_numpy() if hasattr(tr, "to_numpy") else tr],
              y[tr.to_numpy() if hasattr(tr, "to_numpy") else tr])

    shipped = np.array(E.CLASSES)[E.MODEL.predict_proba(X).argmax(1)]
    refit = np.array(m.classes_)[m.predict_proba(X).argmax(1)]
    agree = (shipped == refit).mean()
    print(f"\n  refit agrees with models/outcome.pkl on {agree*100:.2f}% of rows")
    if agree < 0.95:
        print("  -- the recipe here has drifted from the one that produced the shipped model")
else:
    print("\nverifying the shipped model rather than refitting")

P = E.MODEL.predict_proba(X)
pred = np.array(E.CLASSES)[P.argmax(1)]
te = (DS.split == "test").to_numpy() if "split" in DS else np.ones(len(DS), bool)
print(f"  argmax accuracy on {'test' if 'split' in DS else 'all'}: "
      f"{(pred[te] == y[te]).mean()*100:.2f}%   n={te.sum():,}")
print(f"  majority class would give: {(y[te] == 'SUCCESS').mean()*100:.2f}%")
print(f"  expected utility, mean: {(P @ E.UTIL_VEC).mean():+.4f}")


32,532 rows, 8,133 fruit

verifying the shipped model rather than refitting
  argmax accuracy on all: 94.87%   n=32,532
  majority class would give: 84.58%
  expected utility, mean: +0.8165


## 2. Layer 1a — the settled-pose dynamics

Which surviving neighbour is still tilted once the scene comes to rest after a pick.

The physics answers most of it. Where a pick leaves one survivor there is nothing holding it up
and it hangs plumb: across those rows the settled lean averages 0.004° and none exceeds five. The
classifier is for the rows where two or three survivors remain in contact, and there about twelve
per cent stay tilted.

Grouping is by tree. Thirty picks from one canopy share a geometry and their transitions are not
independent, so splitting rows at random would put the same tree on both sides and report a score
that is partly memorisation.


In [3]:
TXD = pd.read_csv(TX/"transitions.csv").drop_duplicates(
    ["tree", "step", "neighbour"], keep="last")
TXD["leaning"] = (TXD.post_lean_deg > 5).astype(int)

FEATS_DYN = E.DYN["features"] if E.DYN else [
    "pre_lean_deg", "pre_ax_x", "pre_ax_y", "pre_ax_z", "pre_bearing_cos", "pre_bearing_sin",
    "pre_along", "pre_radial", "pre_dist", "aperture", "dim_mm", "n_surv", "n_units"]

FIT = TXD[TXD.n_surv >= 2].dropna(subset=FEATS_DYN + ["leaning"]).reset_index(drop=True)
Xd, yd, gd = (FIT[FEATS_DYN].to_numpy(float), FIT.leaning.to_numpy(int),
              FIT.tree.to_numpy())

print(f"{len(TXD):,} transitions; {len(FIT):,} with two or more survivors, "
      f"base rate {yd.mean()*100:.1f}%")
print("\nby survivor count -- one survivor is decided, not predicted\n")
print(TXD.groupby("n_surv").agg(n=("leaning", "size"), leaning=("leaning", "mean"),
                                settled=("post_lean_deg", "mean"),
                                settled_max=("post_lean_deg", "max")).round(4).to_string())

if RUN_TRAIN_DYNAMICS:
    from catboost import CatBoostClassifier
    from sklearn.model_selection import GroupKFold
    from sklearn.metrics import roc_auc_score, average_precision_score

    oof = np.zeros(len(FIT))
    with timed("dynamics model"):
        for tr_, te_ in GroupKFold(n_splits=5).split(Xd, yd, gd):
            m = CatBoostClassifier(iterations=600, learning_rate=0.05, depth=6, verbose=0,
                                   random_seed=0, allow_writing_files=False)
            m.fit(Xd[tr_], yd[tr_])
            oof[te_] = m.predict_proba(Xd[te_])[:, 1]
        final = CatBoostClassifier(iterations=600, learning_rate=0.05, depth=6, verbose=0,
                                   random_seed=0, allow_writing_files=False).fit(Xd, yd)

    auc, ap = roc_auc_score(yd, oof), average_precision_score(yd, oof)
    print(f"\n  AUC {auc:.4f}   AP {ap:.4f}   (shipped: "
          f"{E.DYN['cv_auc']:.4f} / {E.DYN['cv_ap']:.4f})")
    print(f"  pre_lean_deg alone: AUC "
          f"{roc_auc_score(yd, FIT.pre_lean_deg):.4f} -- most of the signal is persistence,")
    print("  and the rest is the arrangement the target was sitting in")

    dst = MODELS/"dynamics.joblib"
    assert not dst.exists(), f"{dst.name} exists -- move it aside rather than overwrite"
    import joblib
    joblib.dump(dict(model=final, name="CatBoost", task="binary", features=FEATS_DYN,
                     target="post_lean_deg > 5", fitted_on="n_surv >= 2 only",
                     n_train=int(len(FIT)), base_rate=float(yd.mean()),
                     cv_auc=float(auc), cv_ap=float(ap)), dst)
else:
    print(f"\nshipped model: AUC {E.DYN['cv_auc']:.4f}, AP {E.DYN['cv_ap']:.4f}, "
          f"base rate {E.DYN['base_rate']*100:.1f}%")
    print(f"  fitted on {E.DYN['n_train']:,} rows, {E.DYN['fitted_on']}")


2,630 transitions; 2,531 with two or more survivors, base rate 12.0%

by survivor count -- one survivor is decided, not predicted

           n  leaning  settled  settled_max
n_surv                                     
1         99   0.0000   0.0042       0.1524
2        416   0.1346   2.6504      73.8496
3       2115   0.1173   2.4232      64.7624

shipped model: AUC 0.9299, AP 0.7612, base rate 12.0%
  fitted on 2,531 rows, n_surv >= 2 only


## 3. Layer 2 — ordering inside a cluster

Enumerate every order within a cluster, score each against the outcome model with the state
updated after each pick, and compare the best against what a greedy rule would have done.

This one is measured and then left out of the chain. If exhaustive search beats greedy by
almost nothing, there is no reason to carry it — and reporting that is more useful than shipping
a component that changes nothing.


In [4]:
from itertools import permutations

CLUSTER_MAX = 4          # beyond this the enumeration stops being cheap and clusters are rare


def cluster_orders(geom, members, cycle=E.PICK_SECONDS):
    '''Expected utility of every order of `members`, and of the greedy rate rule.'''
    best, greedy = -np.inf, None
    for order in permutations(members):
        st = E.TreeState(geom)
        tot = 0.0
        for j in order:
            tot += float(st.probabilities(j) @ E.UTIL_VEC)
            st.remove(j)
        best = max(best, tot)

    st = E.TreeState(geom)
    tot, taken = 0.0, []
    for _ in members:
        u = st.utilities()
        cand = [j for j in members if j not in taken]
        j = max(cand, key=lambda k: u[k])
        tot += float(st.probabilities(j) @ E.UTIL_VEC)
        st.remove(j); taken.append(j)
    return best, tot


rows, t0 = [], time.time()
for tid in list(TREES_EVAL)[:10]:
    geom, _, _ = E.tree_cache(tid, HALF_X, LIFT)
    seen = set()
    for i in range(geom.n):
        if i in seen:
            continue
        grp = [int(k) for k in np.where(geom.WITHIN[i])[0]] + [i]
        grp = sorted(set(grp))
        if not (2 <= len(grp) <= CLUSTER_MAX) or any(k in seen for k in grp):
            continue
        seen.update(grp)
        b, g = cluster_orders(geom, grp)
        rows.append(dict(tree=tid, size=len(grp), best=b, greedy=g, gap=b-g))

CL = pd.DataFrame(rows)
print(f"{len(CL):,} clusters over 10 trees, {time.time()-t0:.1f} s\n")
print(CL.groupby("size").agg(n=("gap", "size"), matched=("gap", lambda s: (s < 1e-9).mean()),
                             mean_gap=("gap", "mean"), max_gap=("gap", "max")).round(4).to_string())
print(f"\n  greedy reaches the optimum on {(CL.gap < 1e-9).mean()*100:.1f}% of clusters")
print(f"  mean gap {CL.gap.mean():.4f} expected utility -- on a shift of ~235 fruit")
print("  Layer 2 stays a benchmark. Nothing downstream calls it.")


239 clusters over 10 trees, 24.5 s

       n  matched  mean_gap  max_gap
size                                
2     80   0.8875    0.0082   0.5956
3     90   0.5778    0.0369   1.1075
4     69   0.1594    0.0267   0.5420

  greedy reaches the optimum on 56.1% of clusters
  mean gap 0.0244 expected utility -- on a shift of ~235 fruit
  Layer 2 stays a benchmark. Nothing downstream calls it.


## 4. Layer 3 — planning a tree

Two networks. The **station planner** chooses where the base stops, decoding one position at a
time and marking which fruit each stop covers so the next choice sees what is left — the first
attempt scored all positions at once and picked three that sat within 15 cm of each other. The
**pick policy** chooses which fruit to take next and when to leave, with the outcome model's
probabilities and the movement cost both as input.

**Both live in `src/planner.py`, and so does everything built on top of them.** This logic was
written into three notebooks before it was written into a module, and the three drifted: one
still decoded three stations, one had no sweep stage, and the figures they produced could not be
put in the same table. Re-implementing it has cost this project more than any other single
mistake, so there is now one copy and the notebooks import it.

Training is not reproduced inline. The pick policy took six revisions to beat the heuristic it
replaced, and each fixed something specific — a self-baseline that pinned the sign of the
advantage, a summed log-probability that made stopping optimal, a missing denominator in the
reward. The notebooks that produced these weights are in `archive/training/`.

**The normalisation is the trap, and the module now guards it.** `FMU` and `FSD` have to be
computed the way they were at training time; computing them any other way collapses the policy
to an immediate stop, which looks like a broken model rather than a broken constant. That has
been diagnosed as a weak model three times. `planner.load()` checks the constants against what
training recorded and raises rather than returning a network that will silently stop.


In [5]:
info = PL.load(ROOT)
print(f"planner {info['planner']}")
print(f"policy  {info['policy']}   scaling {info['scaling']}")
print(f"  move_secs {info['move_secs'][0]:.3f} / {info['move_secs'][1]:.3f}"
      f"   (training recorded 385.324 / 485.485)")
print(f"\nconfiguration: {PL.K_STATIONS} stops, threshold {PL.THRESHOLD}, "
      f"sweep {'on' if PL.SWEEP else 'off'}, arm {PL.HALF_X} x {PL.LIFT} m")
print(f"planner parameters {sum(p.numel() for p in PL.PLANNER.parameters()):,}")
print(f"policy  parameters {sum(p.numel() for p in PL.POLICY.parameters()):,}")




c:\python\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


planner station_planner_k20.pt
policy  pick_policy.pt   scaling recomputed
  move_secs 385.324 / 485.485   (training recorded 385.324 / 485.485)

configuration: 20 stops, threshold 0.0, sweep on, arm 0.2 x 0.6 m
planner parameters 252,769
policy  parameters 648,320


## 5. The chain

Detections in, a harvest plan out: where the base stops, in what order, what to take from each
stop, and what to leave.

Stops come in two stages. The planner decodes twenty, which reaches about 98% of what the arm can
touch with the base free to move. A **sweep stage** then adds one stop for each fruit those
twenty missed — fruit that sit inside the reach but only in positions that add nothing else, so
any rule that stops when the marginal gain hits zero walks past them. They cost a stop apiece and
about 35 s each against 25 s for the rest, and they are the only part of the shortfall that
belongs to the planner rather than to the detector or the arm.

The threshold is zero: everything the stops reach is attempted. That is the configuration for a
grower clearing a finite block, where a fruit the robot skips is not saved but handed to a person.


In [6]:
DEMO = list(TREES_EVAL)[3]

case = PL.tree_case(DEMO)
plan = PL.plan_tree(case)
s = plan.attrs["summary"]

print(f"tree {DEMO}\n")
print(f"  on the tree            {s['on_tree']:>6}")
print(f"  the robot can reach    {s['robot_reach']:>6}")
print(f"  stops                  {s['stops']:>6}"
      f"   ({s['stage1_stops']} from the planner + {s['sweep_stops']} sweep)")
print(f"  in the plan            {s['in_plan']:>6}"
      f"   {s['in_plan']/s['robot_reach']*100:.0f}% of reachable")
print(f"  attempted              {s['attempts']:>6}")
print(f"  expected premium       {s['exp_success']:>6.1f}")
print(f"  expected utility       {s['exp_utility']:>6.1f}")
print(f"  time on the tree       {s['seconds']:>6.0f} s"
      f"   = {s['seconds']/60:.1f} min   ({s['travel']:.0f} s moving)")
print(f"  left for a person      {s['left_reachable']:>6}   within reach, not planned")

print("\nfirst twelve picks\n")
print(plan.head(12).to_string(index=False))



tree 23

  on the tree               120
  the robot can reach        68
  stops                      24   (20 from the planner + 4 sweep)
  in the plan                68   100% of reachable
  attempted                  68
  expected premium         56.1
  expected utility         52.2
  time on the tree         1308 s   = 21.8 min   (356 s moving)
  left for a person           0   within reach, not planned

first twelve picks

 step  stop     x  mast  fruit  height  p_success  exp_utility  secs
    1    21  0.60  4.48    104   4.535     0.9969       0.9961  14.0
    2     6 -0.40  2.23     77   2.259     0.9954       0.9943  46.3
    3     8 -0.30  3.53      8   4.066     0.9951       0.9939  69.3
    4     8 -0.30  3.53     52   3.586     0.9944       0.9930  83.3
    5    19  0.45  2.08     69   2.313     0.9941       0.9926 109.5
    6    19  0.45  2.08     88   2.663     0.9936       0.9921 123.5
    7     2 -0.75  2.13     79   2.133     0.9935       0.9916 141.8
    8    10 -0.3

## 6. What each component contributes

One swap at a time, on the same trees, so the pieces can be told apart. Full evaluation — thirty
seeds, paired comparisons, generalisation — is `03_evaluation`; this is the component view.

`chooser` swaps the trained station planner for coverage search, `picker` swaps the rate rule for
the trained pick policy, and `sweep` turns the second stage off. Each is a keyword on the same
function, which is the point of having the module.


In [7]:
SUBSET = list(TREES_EVAL)[:10]

rows = []
for label, kw in (
        ("planner + rule + sweep",  dict()),                            # shipping
        ("planner + rule, no sweep", dict(sweep=False)),
        ("search  + rule + sweep",  dict(chooser="greedy")),
        ("planner + policy + sweep", dict(picker="policy")),
        ("three stops, threshold 0.9",
         dict(k=3, threshold=0.9, chooser="greedy", sweep=False)),      # the earlier operating point
):
    t0 = time.time()
    S = PL.plan_summary(SUBSET, **kw)
    rows.append(dict(configuration=label, stops=S.stops.mean(),
                     in_plan=S.in_plan.mean(), attempts=S.attempts.mean(),
                     premium=S.exp_success.mean(), utility=S.exp_utility.mean(),
                     minutes=S.seconds.mean()/60, ran=round(time.time()-t0, 1)))

AB = pd.DataFrame(rows)
print(f"{len(SUBSET)} trees, per tree -- expected values, not a shift figure; "
      f"see 03_evaluation\n")
print(AB.round(2).to_string(index=False))

print("\n  The sweep is worth a fraction of a fruit and costs a stop; coverage search reaches")
print("  the same place as the planner once the sweep is on, but needs more sweep stops to get")
print("  there. The policy takes about two thirds of the fruit in half the time with a")
print("  twentieth of the knocked neighbours -- a different operating point, not a worse one,")
print("  and the report gives both.")


10 trees, per tree -- expected values, not a shift figure; see 03_evaluation

             configuration  stops  in_plan  attempts  premium  utility  minutes  ran
    planner + rule + sweep   21.6     64.4      64.4    50.67    46.08    21.15  5.3
  planner + rule, no sweep   20.0     62.6      62.6    49.48    45.09    20.65  5.1
    search  + rule + sweep   22.2     64.4      64.4    50.67    46.08    21.14  4.3
  planner + policy + sweep   21.6     64.4      34.6    33.73    33.54     8.81  6.1
three stops, threshold 0.9    3.0     20.2      14.5    14.23    14.18     4.35  3.3

  The sweep is worth a fraction of a fruit and costs a stop; coverage search reaches
  the same place as the planner once the sweep is on, but needs more sweep stops to get
  there. The policy takes about two thirds of the fruit in half the time with a
  twentieth of the knocked neighbours -- a different operating point, not a worse one,
  and the report gives both.


## 7. What it cost

Training times, and the plan the chain emits. A model that takes forty minutes on a laptop is a
different proposition from one that takes forty seconds, and neither number is visible in an
accuracy table.


In [8]:
if TIMING:
    print("this session\n")
    for k, v in TIMING.items():
        print(f"  {k:<22} {v:8.1f} s")
else:
    print("nothing was trained this session; the flags at the top are all False\n")

# Measured, not estimated. An earlier version of this cell carried figures I had guessed at,
# and two of them were out by an order of magnitude.
print("measured on the runs that produced the shipped weights\n")
print(f"  {'outcome model (LightGBM)':<32} ~1 min")
print(f"  {'dynamics (CatBoost, 5 folds)':<32} ~10 s")
print(f"  {'station planner (Transformer, RL)':<32} 11 min")
print(f"  {'pick policy (Transformer, RL)':<32} 2 h")
print(f"  {'dataset generation (6 shards)':<32} 5-7 h")

spec = dict(trees=TREES_USED,
            arm=dict(half_x=PL.HALF_X, lift=PL.LIFT),
            stations=PL.K_STATIONS, threshold=PL.THRESHOLD, sweep=PL.SWEEP,
            picker="rule", dynamics=bool(E.DYN),
            components=dict(outcome="models/outcome.pkl",
                            dynamics="models/dynamics.joblib",
                            station_planner="models/station_planner_k20.pt",
                            pick_policy="models/pick_policy.pt",
                            planner_module="src/planner.py"),
            demo=dict(tree=int(s["tree"]),
                      stops=int(s["stops"]), stage1_stops=int(s["stage1_stops"]),
                      sweep_stops=int(s["sweep_stops"]),
                      on_tree=int(s["on_tree"]), robot_reach=int(s["robot_reach"]),
                      in_plan=int(s["in_plan"]), attempts=int(s["attempts"]),
                      expected_premium=round(float(s["exp_success"]), 2),
                      expected_utility=round(float(s["exp_utility"]), 2),
                      left_for_a_person=int(s["left_reachable"]),
                      seconds=round(float(s["seconds"]), 1)))
(MODELS/"pipeline.json").write_text(json.dumps(spec, indent=1))
print(f"\nwritten {MODELS/'pipeline.json'}")
print(json.dumps(spec, indent=1))


nothing was trained this session; the flags at the top are all False

measured on the runs that produced the shipped weights

  outcome model (LightGBM)         ~1 min
  dynamics (CatBoost, 5 folds)     ~10 s
  station planner (Transformer, RL) 11 min
  pick policy (Transformer, RL)    2 h
  dataset generation (6 shards)    5-7 h

written c:\aipick\git\models\pipeline.json
{
 "trees": "trees_measured_pose.csv",
 "arm": {
  "half_x": 0.2,
  "lift": 0.6
 },
 "stations": 20,
 "threshold": 0.0,
 "sweep": true,
 "picker": "rule",
 "dynamics": true,
 "components": {
  "outcome": "models/outcome.pkl",
  "dynamics": "models/dynamics.joblib",
  "station_planner": "models/station_planner_k20.pt",
  "pick_policy": "models/pick_policy.pt",
  "planner_module": "src/planner.py"
 },
 "demo": {
  "tree": 23,
  "stops": 24,
  "stage1_stops": 20,
  "sweep_stops": 4,
  "on_tree": 120,
  "robot_reach": 68,
  "in_plan": 68,
  "attempts": 68,
  "expected_premium": 56.1,
  "expected_utility": 52.17,
  "left_

### What comes next

`03_evaluation` measures this chain: the outcome model against held-out fruit, the configurations
against each other over thirty paired seeds, and the learned components on the trees they were
fitted on against the ones they were not.
